# TCN + GRU + XGBoost -- CMAPSS Seq2Seq (Engine-Level) with Outlier Cleaning

**Mimarideki Yenilikler (Yeni Seq2Seq Yaklaşımı):**
1. **Outlier Cleaning (IQR Clipping):** Standardizasyondan önce anlık sensör hataları (noise ve uç değerler) IQR sınırları dahilinde tıraşlanarak (clipping) temizlenir. Bu sayede model sapmalara karşı çok daha dirençli hale gelir.
2. **Engine-Level Seq2Seq:** Sliding window yerine her bir motorun tüm ömrü (padding işlemi ile eşitlenerek) bütünsel bir sekans (sequence) olarak modele verilir. Bu işlem GRU'nun hafıza gücünü maksimize eder.
3. **Causal GRU:** Seq2Seq eğitiminde, her bir zaman adımında (`t`) tahminde bulunurken gelecekten (`t+1, t+2...`) bilgi sızıntısını (leakage) önlemek için BiGRU yerine tek yönlü (Causal) GRU kullanılmıştır. TCN zaten *causal convolution* yapmaktadır. Bu sayede test setinde operasyonel şartlara %100 uyumlu bir yapı kurulmuştur.
4. **Masked Loss:** Sekanslar padding ile uzatıldığından, modelin kayıp (loss) hesaplamasında sadece gerçek geçerli adımlar hesaba katılır. *Masked Focal Loss* ve *Masked MSE* kullanılarak padding'in modeli yanıltması sıfırlanır.
5. **High Recall Focus:** F1 > 0.9 hedefi gözetilirken, hata tespitinde en kritik nokta olan Recall (Duyarlılık) değerini yüksek tutmak için threshold seçiminde `min_recall_floor` kullanılmış ve Focal Loss'ta pozitif sınıf baskısı artırılmıştır.

In [ ]:
import numpy as np
import pandas as pd
import os, warnings, random
from pathlib import Path
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, fbeta_score, recall_score, roc_auc_score,
    classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score, roc_curve,
    make_scorer, precision_recall_fscore_support
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold

import xgboost as xgb
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

Device: cuda


## Configuration (Seq2Seq & Outliers)

In [ ]:
BASE = './data'

CFG = {
    'dataset'          : 'FD001',   
    'rul_threshold'    : 30,
    'val_engine_frac'  : 0.15,

    # --- Seq2Seq & Outlier Params ---
    'outlier_multiplier' : 1.5,
    'rolling_window_size': 15,

    # --- TCN ---
    'tcn_channels'     : [64, 128, 128, 64],
    'tcn_kernel'       : 3,
    'tcn_dropout'      : 0.2,

    # --- Causal GRU (Seq2Seq) ---
    'gru_hidden'       : 128,
    'gru_layers'       : 2,
    'gru_dropout'      : 0.3,

    # --- Embedding ---
    'embed_dim'        : 128,

    # --- Training ---
    'epochs'           : 60,
    'batch_size'       : 32,         # Smaller because whole sequences are passed
    'lr'               : 1e-3,
    'weight_decay'     : 1e-4,
    'patience'         : 12,
    'focal_gamma'      : 2.0,
    'recon_weight'     : 0.1,        

    # --- Recall-boost ---
    'pos_weight_boost' : 2.5,
    'recall_beta'      : 2.0,
    
    # --- Thresholding ---
    'min_precision'    : 0.70,
    'min_recall_floor' : 0.85,       # Focus on high recall!
    'f1_tolerance'     : 0.005,

    # --- XGBoost ---
    'xgb_rounds'       : 800,
    'xgb_lr'           : 0.03,
    'xgb_depth'        : 6,
    'xgb_subsample'    : 0.8,
    'xgb_colsample'    : 0.8,
    'xgb_min_child'    : 3,
    'xgb_alpha'        : 0.1,
    'xgb_lambda'       : 1.0,
    'xgb_es_rounds'    : 40,
    'grid_cv_folds'    : 3,
    'grid_n_jobs'      : -1,
}

## Data Loading, Outlier Cleaning & Seq2Seq Masking

In [ ]:
COLUMNS = (
    ['unit_number', 'time_in_cycles']
    + [f'op_{i}' for i in range(1, 4)]
    + [f's{i}'  for i in range(1, 22)]
)

DROP_SENSORS = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']

def resolve_data_dir(data_dir):
    candidates = [
        Path(data_dir),
        Path.cwd() / data_dir,
        Path('/content/drive/MyDrive/data'),
        Path('/content/data'),
    ]
    for path in candidates:
        if (path / 'train_FD001.txt').exists():
            return path
    checked = ', '.join(str(p) for p in candidates)
    raise FileNotFoundError('CMAPSS data directory not found. Checked: ' + checked)

def load_raw(fd, data_dir):
    data_dir = resolve_data_dir(data_dir)
    train = pd.read_csv(data_dir / f'train_{fd}.txt', sep=r'\s+', header=None, names=COLUMNS)
    test = pd.read_csv(data_dir / f'test_{fd}.txt', sep=r'\s+', header=None, names=COLUMNS)
    rul = pd.read_csv(data_dir / f'RUL_{fd}.txt', sep=r'\s+', header=None, names=['RUL'])
    return train, test, rul

def add_train_rul(df):
    max_c = df.groupby('unit_number')['time_in_cycles'].max()
    df = df.join(max_c.rename('max_c'), on='unit_number')
    df['RUL'] = df['max_c'] - df['time_in_cycles']
    return df.drop('max_c', axis=1)

def add_test_rul(test_df, rul_df):
    test_df = test_df.copy()
    max_c = test_df.groupby('unit_number')['time_in_cycles'].max().reset_index()
    max_c.columns = ['unit_number', 'max_c']
    rul = rul_df.copy()
    rul['unit_number'] = np.arange(1, len(rul) + 1)
    max_c = max_c.merge(rul, on='unit_number', how='left')
    max_c['final_cycle'] = max_c['max_c'] + max_c['RUL']
    test_df = test_df.merge(max_c[['unit_number', 'final_cycle']], on='unit_number', how='left')
    test_df['RUL'] = test_df['final_cycle'] - test_df['time_in_cycles']
    return test_df.drop('final_cycle', axis=1)

def get_features(df):
    drop = ['unit_number', 'time_in_cycles', 'RUL'] + DROP_SENSORS
    return [c for c in df.columns if c not in drop]

def split_train_val(train_df, val_frac, seed=42):
    units = train_df['unit_number'].unique()
    rng   = np.random.RandomState(seed)
    val_units = rng.choice(units, size=int(len(units) * val_frac), replace=False)
    val_mask  = train_df['unit_number'].isin(val_units)
    return train_df[~val_mask].copy(), train_df[val_mask].copy()

def remove_outliers_iqr(df, columns, multiplier=1.5):
    """IQR clipping for robust scaling and noise suppression."""
    df_clean = df.copy()
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        df_clean[col] = np.clip(df_clean[col], lower_bound, upper_bound)
    return df_clean

def make_engine_sequences(df, feat_cols, threshold):
    """Seq2Seq: Full engine lifecycle with zero-padding and masking."""
    X_list, y_list, mask_list, eng_list = [], [], [], []
    max_len = df.groupby('unit_number').size().max()
    
    for uid, group in df.groupby('unit_number'):
        group = group.sort_values('time_in_cycles')
        arr = group[feat_cols].values.astype(np.float32)
        labels = (group['RUL'].values <= threshold).astype(np.int64)
        
        T = len(arr)
        pad_len = max_len - T
        
        if pad_len > 0:
            X_pad = np.pad(arr, ((0, pad_len), (0, 0)), 'constant', constant_values=0)
            y_pad = np.pad(labels, (0, pad_len), 'constant', constant_values=0)
            mask  = np.pad(np.ones(T, dtype=np.float32), (0, pad_len), 'constant', constant_values=0)
        else:
            X_pad = arr
            y_pad = labels
            mask  = np.ones(T, dtype=np.float32)
            
        X_list.append(X_pad)
        y_list.append(y_pad)
        mask_list.append(mask)
        eng_list.append(int(uid))
        
    return (np.array(X_list, dtype=np.float32), 
            np.array(y_list, dtype=np.int64), 
            np.array(mask_list, dtype=np.float32),
            np.array(eng_list, dtype=np.int64))

# Load
FD = CFG['dataset']
train_raw, test_raw, rul_raw = load_raw(FD, BASE)
train_raw = add_train_rul(train_raw)
test_raw  = add_test_rul(test_raw, rul_raw)

tr_df, val_df = split_train_val(train_raw, CFG['val_engine_frac'])
feat_cols = get_features(tr_df)

# 1. Outlier Cleaning (Fit only on Train to avoid leakage)
print('Applying IQR Outlier Clipping...')
tr_df = remove_outliers_iqr(tr_df, feat_cols, CFG['outlier_multiplier'])
val_df = remove_outliers_iqr(val_df, feat_cols, CFG['outlier_multiplier'])
test_raw = remove_outliers_iqr(test_raw, feat_cols, CFG['outlier_multiplier'])

# 2. Standardise
scaler = StandardScaler()
tr_df[feat_cols]    = scaler.fit_transform(tr_df[feat_cols])
val_df[feat_cols]   = scaler.transform(val_df[feat_cols])
test_raw[feat_cols] = scaler.transform(test_raw[feat_cols])

# 3. Engine Sequences (Seq2Seq)
TH = CFG['rul_threshold']
X_tr,  y_tr,  m_tr,  eng_tr  = make_engine_sequences(tr_df,  feat_cols, TH)
X_val, y_val, m_val, eng_val = make_engine_sequences(val_df, feat_cols, TH)
X_te,  y_te,  m_te,  eng_te  = make_engine_sequences(test_raw, feat_cols, TH)

print(f'Train Seq : {X_tr.shape} | mask valid: {int(m_tr.sum())} steps')
print(f'Val Seq   : {X_val.shape} | mask valid: {int(m_val.sum())} steps')
print(f'Test Seq  : {X_te.shape} | mask valid: {int(m_te.sum())} steps')

Applying IQR Outlier Clipping...
Train Seq : (85, 362, 17) | mask valid: 17569 steps
Val Seq   : (15, 267, 17) | mask valid: 3062 steps
Test Seq  : (100, 303, 17) | mask valid: 13096 steps


## Model: TCN + Causal GRU (Seq2Seq)

In [ ]:
class CausalConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, dilation, dropout):
        super().__init__()
        self.pad = (kernel - 1) * dilation
        self.conv1 = nn.utils.weight_norm(nn.Conv1d(in_ch, out_ch, kernel, padding=self.pad, dilation=dilation))
        self.conv2 = nn.utils.weight_norm(nn.Conv1d(out_ch, out_ch, kernel, padding=self.pad, dilation=dilation))
        self.drop  = nn.Dropout(dropout)
        self.relu  = nn.ReLU()
        self.res   = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        # Strictly causal: keep first T elements, drop right padding
        out = self.relu(self.conv1(x)[:, :, :x.size(2)])
        out = self.drop(out)
        out = self.relu(self.conv2(out)[:, :, :x.size(2)])
        out = self.drop(out)
        res = x if self.res is None else self.res(x)
        return self.relu(out + res)

class TCN(nn.Module):
    def __init__(self, in_ch, channels, kernel, dropout):
        super().__init__()
        layers = []
        for i, out_ch in enumerate(channels):
            layers.append(CausalConvBlock(in_ch, out_ch, kernel, dilation=2**i, dropout=dropout))
            in_ch = out_ch
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.permute(0, 2, 1) 
        x = self.net(x)
        return x.permute(0, 2, 1)

class TCNGRU_Seq2Seq(nn.Module):
    """TCN + Causal GRU for Seq2Seq RUL Prediction.\n       Bidirectional GRU removed to prevent future data leakage."""
    def __init__(self, n_feat, cfg):
        super().__init__()
        self.tcn = TCN(n_feat, cfg['tcn_channels'], cfg['tcn_kernel'], cfg['tcn_dropout'])
        tcn_out  = cfg['tcn_channels'][-1]

        self.gru = nn.GRU(
            input_size    = tcn_out,
            hidden_size   = cfg['gru_hidden'],
            num_layers    = cfg['gru_layers'],
            batch_first   = True,
            bidirectional = False, # Uni-directional to remain causal
            dropout       = cfg['gru_dropout'] if cfg['gru_layers'] > 1 else 0,
        )
        gru_out = cfg['gru_hidden']

        self.proj = nn.Sequential(
            nn.Linear(gru_out, cfg['embed_dim']),
            nn.LayerNorm(cfg['embed_dim']),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.cls_head   = nn.Linear(cfg['embed_dim'], 1)
        self.recon_head = nn.Sequential(
            nn.Linear(cfg['embed_dim'], 64),
            nn.ReLU(),
            nn.Linear(64, n_feat),
        )

    def embed(self, x):
        t, _ = self.gru(self.tcn(x))
        return self.proj(t)

    def forward(self, x):
        z = self.embed(x)
        return self.cls_head(z).squeeze(-1), self.recon_head(z), z

n_feat = X_tr.shape[2]
model  = TCNGRU_Seq2Seq(n_feat, CFG).to(DEVICE)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Trainable parameters: 443,538


## Masked Loss + Training Loop

In [ ]:
class MaskedFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets, mask):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets.float(), pos_weight=self.pos_weight, reduction='none')
        prob = torch.sigmoid(logits)
        pt   = torch.where(targets == 1, prob, 1 - prob)
        focal = ((1 - pt) ** self.gamma * bce)
        return (focal * mask).sum() / (mask.sum() + 1e-9)

def masked_mse_loss(recon, targets, mask):
    mse = F.mse_loss(recon, targets, reduction='none').mean(dim=-1)
    return (mse * mask).sum() / (mask.sum() + 1e-9)

def get_seq_loader(X, y, mask, batch_size, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y), torch.from_numpy(mask))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=True)

y_tr_valid = y_tr[m_tr == 1]
raw_pos_ratio = float((y_tr_valid == 0).sum() / (y_tr_valid == 1).sum())
boosted_pw    = raw_pos_ratio * CFG['pos_weight_boost']
pos_w = torch.tensor([boosted_pw], dtype=torch.float32).to(DEVICE)

cls_criterion   = MaskedFocalLoss(gamma=CFG['focal_gamma'], pos_weight=pos_w)
RECON_W         = CFG['recon_weight']

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-5)

tr_loader  = get_seq_loader(X_tr,  y_tr, m_tr,  CFG['batch_size'], shuffle=True)
val_loader = get_seq_loader(X_val, y_val, m_val, CFG['batch_size'], shuffle=False)

BETA = CFG['recall_beta']

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_trues = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb, mb in loader:
            xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
            logits, recon, _ = model(xb)
            
            cls_loss   = cls_criterion(logits, yb, mb)
            recon_loss = masked_mse_loss(recon, xb, mb)
            loss = cls_loss + RECON_W * recon_loss
            
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
            total_loss += loss.item() * len(xb)
            
            prob = torch.sigmoid(logits).detach().cpu().numpy()
            yb_np = yb.cpu().numpy()
            mb_np = mb.cpu().numpy()
            
            for i in range(len(prob)):
                valid_len = int(mb_np[i].sum())
                all_preds.extend(prob[i, :valid_len])
                all_trues.extend(yb_np[i, :valid_len])
                
    all_preds, all_trues = np.array(all_preds), np.array(all_trues)
    fb  = fbeta_score(all_trues, (all_preds > 0.5).astype(int), beta=BETA, zero_division=0)
    auc = roc_auc_score(all_trues, all_preds) if len(np.unique(all_trues)) > 1 else 0.5
    return total_loss / len(loader.dataset), fb, auc

print('Seq2Seq Hybrid Training (Masked Focal + MSE)...')
best_val_fb, best_state, no_improve = -1.0, None, 0

for ep in range(1, CFG['epochs'] + 1):
    tr_loss,  tr_fb,  _       = run_epoch(tr_loader,  train=True)
    val_loss, val_fb, val_auc = run_epoch(val_loader, train=False)
    scheduler.step()
    
    if val_fb > best_val_fb:
        best_val_fb = val_fb
        best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve  = 0
    else:
        no_improve += 1
        
    if ep % 5 == 0 or ep == 1:
        print(f'Epoch {ep:2d} | tr_loss={tr_loss:.4f} tr_F{BETA:.0f}={tr_fb:.4f} | '
              f'val_loss={val_loss:.4f} val_F{BETA:.0f}={val_fb:.4f} val_auc={val_auc:.4f}')
    if no_improve >= CFG['patience']:
        print(f'Early stopping at epoch {ep}')
        break

model.load_state_dict(best_state)
print(f'Best Validation F{BETA:.0f}: {best_val_fb:.4f}')

Seq2Seq Hybrid Training (Masked Focal + MSE)...
Epoch  1 | tr_loss=0.6559 tr_F2=0.4327 | val_loss=0.4131 val_F2=0.4728 val_auc=0.9848
Epoch  5 | tr_loss=0.1925 tr_F2=0.8498 | val_loss=0.1766 val_F2=0.8763 val_auc=0.9912
Epoch 10 | tr_loss=0.1205 tr_F2=0.9169 | val_loss=0.1072 val_F2=0.9150 val_auc=0.9966
Epoch 15 | tr_loss=0.0973 tr_F2=0.9451 | val_loss=0.0977 val_F2=0.9234 val_auc=0.9965
Epoch 20 | tr_loss=0.0918 tr_F2=0.9518 | val_loss=0.0886 val_F2=0.9259 val_auc=0.9976
Epoch 25 | tr_loss=0.0832 tr_F2=0.9554 | val_loss=0.0924 val_F2=0.9266 val_auc=0.9974
Early stopping at epoch 29
Best Validation F2: 0.9466


## Embedding Extraction & Rolling Semantic Features (Fused for XGBoost)

In [ ]:
def extract_features_and_flatten(X_pad, y_pad, masks, eng_pad, is_test_last=False):
    N, T, C = X_pad.shape
    
    # 1. TCN-GRU Embeddings (B, T, E)
    model.eval()
    all_embs = []
    with torch.no_grad():
        for i in range(0, N, CFG['batch_size']):
            xb = torch.from_numpy(X_pad[i:i+CFG['batch_size']]).to(DEVICE)
            all_embs.append(model.embed(xb).cpu().numpy())
    all_embs = np.concatenate(all_embs, axis=0)
    
    # 2. Causal Rolling Semantic Features (B, T, S)
    all_sem = np.zeros((N, T, C * 6), dtype=np.float32)
    ws = CFG['rolling_window_size']
    for i in range(N):
        valid_len = int(masks[i].sum())
        df_x = pd.DataFrame(X_pad[i, :valid_len, :])
        
        roll = df_x.rolling(window=ws, min_periods=1)
        feats = np.concatenate([
            roll.mean().values,
            roll.std().fillna(0).values,
            roll.min().values,
            roll.max().values,
            df_x.ewm(span=ws, adjust=False).mean().values,
            df_x.diff().fillna(0).values
        ], axis=1)
        all_sem[i, :valid_len, :] = feats
        
    # Fuse -> (N, T, E + S)
    fused = np.concatenate([all_embs, all_sem], axis=2)
    
    # 3. Flatten valid timesteps based on protocol
    out_X, out_y, out_eng = [], [], []
    for i in range(N):
        valid_len = int(masks[i].sum())
        if is_test_last:
            out_X.append(fused[i, valid_len - 1, :])
            out_y.append(y_pad[i, valid_len - 1])
            out_eng.append(eng_pad[i])
        else:
            out_X.append(fused[i, :valid_len, :])
            out_y.append(y_pad[i, :valid_len])
            out_eng.append(np.full(valid_len, eng_pad[i]))
            
    if is_test_last:
        return np.array(out_X), np.array(out_y), np.array(out_eng)
    return np.vstack(out_X), np.concatenate(out_y), np.concatenate(out_eng)

print('Extracting fused Seq2Seq + Causal Rolling features...')
f_tr, y_tr_flat, _ = extract_features_and_flatten(X_tr, y_tr, m_tr, eng_tr)
f_val, y_val_flat, _ = extract_features_and_flatten(X_val, y_val, m_val, eng_val)
f_te_last, y_te_last, eng_te_last = extract_features_and_flatten(X_te, y_te, m_te, eng_te, is_test_last=True)
f_te_all, y_te_all, eng_te_all = extract_features_and_flatten(X_te, y_te, m_te, eng_te, is_test_last=False)

print(f'Train Flat: {f_tr.shape} | Val Flat: {f_val.shape}')
print(f'Test Last: {f_te_last.shape} | Test All (Sliding eq): {f_te_all.shape}')

Extracting fused Seq2Seq + Causal Rolling features...
Train Flat: (17569, 230) | Val Flat: (3062, 230)
Test Last: (100, 230) | Test All (Sliding eq): (13096, 230)


## XGBoost Train

In [ ]:
# GridSearchCV (Train Only)
scale_pos = float((y_tr_flat == 0).sum() / (y_tr_flat == 1).sum())
scale_pos_bst = scale_pos * CFG['pos_weight_boost']
BETA = CFG['recall_beta']

f_beta_scorer = make_scorer(fbeta_score, beta=BETA, zero_division=0)
param_grid = {'max_depth': [4, 6], 'learning_rate': [0.03, 0.05]}

xgb_base = XGBClassifier(objective='binary:logistic', n_estimators=200, scale_pos_weight=scale_pos_bst, tree_method='hist', random_state=42)
grid_search = GridSearchCV(xgb_base, param_grid, scoring=f_beta_scorer, cv=3, n_jobs=CFG['grid_n_jobs'], verbose=1)
grid_search.fit(f_tr, y_tr_flat)
best_params = grid_search.best_params_
print(f'\nBest Params: {best_params}')

# Final XGBoost Train
dtrain = xgb.DMatrix(f_tr, label=y_tr_flat)
dval   = xgb.DMatrix(f_val, label=y_val_flat)

xgb_params = {'objective': 'binary:logistic', 'eval_metric': 'auc', 'eta': best_params['learning_rate'], 'max_depth': best_params['max_depth'], 'scale_pos_weight': scale_pos_bst, 'tree_method': 'hist', 'seed': 42}
xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=CFG['xgb_rounds'], evals=[(dtrain, 'train'), (dval, 'val')], early_stopping_rounds=CFG['xgb_es_rounds'], verbose_eval=50)


Fitting 3 folds for each of 4 candidates, totalling 12 fits

Best Params: {'learning_rate': 0.05, 'max_depth': 4}
[0]	train-auc:0.99359	val-auc:0.98689
[50]	train-auc:0.99934	val-auc:0.99394
[100]	train-auc:0.99983	val-auc:0.99655
[150]	train-auc:0.99998	val-auc:0.99655


## Threshold Selection (High Recall Focus) & Test Evaluation

In [ ]:
# Threshold Selection (Optimize F1, Prioritize Recall > 0.85)
val_probs = xgb_model.predict(dval)

cands = []
for th in np.arange(0.05, 0.95, 0.005):
    y_hat = (val_probs >= th).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_val_flat, y_hat, average='binary', zero_division=0)
    cands.append((th, p, r, f1))

valid = [c for c in cands if c[1] >= CFG['min_precision'] and c[2] >= CFG['min_recall_floor']]
if not valid:
    valid = [c for c in cands if c[1] >= CFG['min_precision']]
    if not valid: valid = cands

best = sorted(valid, key=lambda c: (-c[3], -c[2], c[0]))[0]
best_th, best_p, best_r, best_f1 = best
print(f'VAL Selected Threshold: {best_th:.4f} | F1: {best_f1:.4f} | Precision: {best_p:.4f} | Recall: {best_r:.4f}')

# Test Evaluation
p_last = xgb_model.predict(xgb.DMatrix(f_te_last))
p_all  = xgb_model.predict(xgb.DMatrix(f_te_all))

y_hat_last = (p_last >= best_th).astype(int)
y_hat_all  = (p_all >= best_th).astype(int)

def print_metrics(name, y_t, y_p):
    p, r, f1, _ = precision_recall_fscore_support(y_t, y_p, average='binary', zero_division=0)
    print(f"{name:<20} | F1: {f1:.4f} | Precision: {p:.4f} | Recall: {r:.4f}")

print('\n--- TEST RESULTS ---')
print_metrics('Test Protocol (Last)', y_te_last, y_hat_last)
print_metrics('Test Protocol (All)', y_te_all, y_hat_all)

VAL Selected Threshold: 0.9350 | F1: 0.9229 | Precision: 0.9458 | Recall: 0.9011

--- TEST RESULTS ---
Test Protocol (Last) | F1: 0.8182 | Precision: 0.9474 | Recall: 0.7200
Test Protocol (All)  | F1: 0.7273 | Precision: 0.8667 | Recall: 0.6265
